# Spotify Song Language Detection

This notebook detects the language of songs in the `spotify_411k.parquet` dataset using both lyrics and track titles. It utilizes the `pycld2` library for fast and accurate multi-language detection.

In [2]:
import pandas as pd
import pycld2 as cld2
import re
from tqdm.notebook import tqdm

# Enable progress bars for pandas
tqdm.pandas()

## Detection Logic

The function below combines the track name and lyrics (if available) to detect the language. It returns multiple languages separated by a semicolon if they are significantly present (threshold > 15%).

In [3]:
def detect_languages(row):
    """
    Accurately determines song language based on lyrics and title.
    Returns language codes (e.g., 'en', 'es', 'en;es') or 'unknown'.
    """
    lyrics = str(row.get('lyrics', ''))
    title = str(row.get('track_name', ''))
    
    # Standardize 'None' or empty strings
    if lyrics.lower() == 'none' or not lyrics.strip():
        text = title
    else:
        # We prioritize lyrics by putting them in the context,
        # but include the title to help with very short lyrics or instrumentals with descriptive titles.
        text = f"{title} {lyrics}"
    
    # Basic cleaning to help the detector
    text = re.sub(r'\s+', ' ', text).strip()
    
    # pycld2 requires at least some text and can fail on certain non-UTF8 chars or strictly symbols
    if not text or len(text) < 2:
        return "unknown"
        
    try:
        # cld2.detect returns: (isReliable, textBytesFound, details)
        # details is a tuple of 3: (language_name, language_code, percent, score)
        is_reliable, _, details = cld2.detect(text)
        
        langs = []
        for name, code, percent, score in details:
            # Filter out 'Unknown' (un) and require at least 15% presence for secondary languages
            if code != 'un' and percent > 15:
                langs.append(code)
        
        # If no specific languages found but the first one isn't 'un', use it
        if not langs and details[0][1] != 'un':
            langs = [details[0][1]]
            
        return ";".join(langs) if langs else "unknown"
        
    except Exception:
        # Fallback if cld2 fails on specific encoding issues
        return "unknown"

## Load and Process Data

In [4]:
FILE_PATH = 'spotify_411k.parquet'
print(f"Loading {FILE_PATH}...")
df = pd.read_parquet(FILE_PATH)

print("Detecting languages (this may take a few minutes)...")
df['language'] = df.progress_apply(detect_languages, axis=1)

Loading spotify_411k.parquet...
Detecting languages (this may take a few minutes)...


  0%|          | 0/411000 [00:00<?, ?it/s]

## Inspection and Export

In [5]:
# Display some samples to verify accuracy
print("Sample detections:")
display(df[['track_name', 'language']].sample(20) if len(df) > 20 else df[['track_name', 'language']])

# Show language distribution
print("\nLanguage distribution:")
print(df['language'].value_counts().head(15))

Sample detections:


,track_name,language
85616,Boyz-N-The-Hood - Remix,en
241922,Evening Star,unknown
90933,Let Her Go,en
294953,"Angry, Young and Poor",en
317267,Mt. Abraxas,en
368112,Invoke the Black Oblivion,en
91476,Time to Try Again,en
107929,放過自己,zh-Hant
175390,Te Joseph,pt
150878,3 Words,en



Language distribution:
language
en         266437
unknown     52540
es          25212
pt          18429
fr           6004
de           5329
zh-Hant      3941
ko           3717
ko;en        3142
ja           2555
hi           2237
sv           1771
ru           1735
it           1181
ta            930
Name: count, dtype: int64


## Fallback Detection for "Unknown" Languages

For tracks where  returned "unknown", we use  as a secondary method. If it still can't identify the language, we assume it's likely instrumental and leave it as "unknown".

In [ ]:
from langdetect import detect as lang_detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Ensure consistent results with langdetect
DetectorFactory.seed = 42

def fallback_detect(row):
    # Only process if previous step was unknown
    if row['language'] != 'unknown':
        return row['language']
        
    lyrics = str(row.get('lyrics', ''))
    title = str(row.get('track_name', ''))
    
    if lyrics.lower() == 'none' or not lyrics.strip():
        text = title
    else:
        text = f"{title} {lyrics}"
    
    # Clean text: remove extra whitespaces
    import re
    text = re.sub(r'\s+', ' ', text).strip()
    
    if not text or len(text) < 2:
        return "unknown"
        
    try:
        # returns a single language code
        return lang_detect(text)
    except LangDetectException:
        return "unknown"

# Apply fallback only to unknown rows for efficiency
unknown_mask = df['language'] == 'unknown'
print(f"Applying fallback to {unknown_mask.sum()} unknown rows...")

df.loc[unknown_mask, 'language'] = df[unknown_mask].progress_apply(fallback_detect, axis=1)

# Final check
print("\nUpdated Language distribution:")
print(df['language'].value_counts().head(15))

In [7]:
# Optional: Save the result
df.to_parquet('spotify_411k_with_languages.parquet')